# Notebook 06b — BERTopic: Fresh Run (Clean Outputs)

**Purpose:** Complete, clean rerun of BERTopic topic modeling on the full
1115-article corpus. Replaces nb06 which had incomplete outputs and
inconsistent column naming.

**What this notebook produces:**
- All 30 topics with full keyword lists (input for taxonomy definition)
- Clean topic assignments for all 1115 articles
- Validation set (183 articles) topic distribution
- Topic coherence and diversity metrics
- All saved with consistent column names for downstream notebooks

**Design decisions:**
- BERTopic run on German original text (`content`) — language of the corpus
- Cached embeddings reused if valid — saves ~10 minutes of compute
- Outlier articles (topic=-1) explicitly labelled as OTHER in assignments
- No LLM topic classification in this notebook — that is Notebook 15

**Outputs (saved to Data/Processed/):**
- `topic_assignments_v2.csv` — topic per article, clean column names
- `topic_model_v2` — saved BERTopic model
- `topic_embeddings_v2.npy` — embeddings cache
- `topic_summary_v2.json` — metrics, topic keywords, validation coverage


In [ ]:
# ── CELL 1 : INSTALLATION ────────────────────────────────────────────────────
!pip install -q bertopic
!pip install -q sentence-transformers
!pip install -q umap-learn hdbscan
!pip install -q 'numpy>=2.0'   # must be last


In [ ]:
# ── CELL 2 : IMPORTS & CONFIGURATION ─────────────────────────────────────────

import os, json, pickle, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=False)

ROOT        = Path('/content/drive/MyDrive/thesis')
DATA_PROC   = ROOT / 'Project/Data/Processed'
FIGURES_DIR = ROOT / 'Project/Outputs/Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Paths — new v2 outputs ────────────────────────────────────────────────────
ASSIGNMENTS_OUT  = DATA_PROC / 'topic_assignments_v2.csv'
MODEL_OUT        = DATA_PROC / 'topic_model_v2'
EMBEDDINGS_NEW   = DATA_PROC / 'topic_embeddings_v2.npy'
EMBEDDINGS_OLD   = DATA_PROC / 'topic_embeddings.npy'   # reuse if valid
SUMMARY_OUT      = DATA_PROC / 'topic_summary_v2.json'

# ── BERTopic settings ────────────────────────────────────────────────────────
EMBEDDING_MODEL  = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_TOPIC_SIZE   = 15    # minimum articles per topic
NR_TOPICS        = "auto"
RANDOM_STATE     = 42

print("✓ Configuration loaded")
print(f"  Corpus   : 1115 articles (full)")
print(f"  Embedding: {EMBEDDING_MODEL}")
print(f"  Min topic size: {MIN_TOPIC_SIZE}")


In [ ]:
# ── CELL 3 : LOAD DATA ───────────────────────────────────────────────────────

print("Loading corpus...")
pipeline_df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
print(f"  Shape   : {pipeline_df.shape}")
print(f"  Columns : {[c for c in pipeline_df.columns if c in ['article_id','content','content_en','source','year','year_bin']]}")

# Validate required columns
for col in ['article_id', 'content', 'source', 'year', 'year_bin']:
    assert col in pipeline_df.columns, f"Missing column: {col}"

# Remove articles with empty German text
pipeline_df = pipeline_df[pipeline_df['content'].notna() &
                           (pipeline_df['content'].str.strip() != '')]
print(f"  After removing empty: {len(pipeline_df)} articles")

# Extract texts and metadata
texts      = pipeline_df['content'].tolist()
article_ids= pipeline_df['article_id'].tolist()

print(f"\n  Source distribution:")
print(pipeline_df['source'].value_counts().to_string())
print(f"\n  Year distribution:")
print(pipeline_df['year'].value_counts().sort_index().to_string())

# ── Load validation set IDs ───────────────────────────────────────────────────
qwen_path = DATA_PROC / 'ner_llm_checkpoint.pkl'
with open(qwen_path, 'rb') as f:
    qwen_ckpt = pickle.load(f)
VAL_IDS = set(aid for aid, data in qwen_ckpt.items()
              if isinstance(data, dict)
              and data.get('status') != 'api_error')
print(f"\n  Validation set: {len(VAL_IDS)} articles")


In [ ]:
# ── CELL 4 : EMBEDDINGS ──────────────────────────────────────────────────────
# Reuse cached embeddings if they match the current corpus size.
# Recompute from scratch otherwise.

from sentence_transformers import SentenceTransformer

n_docs = len(texts)

def try_load_embeddings(path: Path, expected_n: int) -> np.ndarray | None:
    if not path.exists():
        return None
    try:
        emb = np.load(str(path))
        if emb.shape[0] == expected_n:
            print(f"  ✓ Loaded cached embeddings: {emb.shape}  ({path.name})")
            return emb
        else:
            print(f"  ✗ Cached embeddings shape {emb.shape} ≠ expected ({expected_n},*)")
            return None
    except Exception as e:
        print(f"  ✗ Could not load {path.name}: {e}")
        return None

print("Checking for cached embeddings...")

# Try new cache first, then old cache
embeddings = try_load_embeddings(EMBEDDINGS_NEW, n_docs)
if embeddings is None:
    embeddings = try_load_embeddings(EMBEDDINGS_OLD, n_docs)

if embeddings is None:
    print(f"  No valid cache found — computing embeddings for {n_docs} articles...")
    print(f"  Model: {EMBEDDING_MODEL}")
    print(f"  Expected time: ~5-8 minutes on T4 GPU")
    embed_model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings  = embed_model.encode(
        texts,
        show_progress_bar = True,
        batch_size        = 64,
        device            = 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    )
    np.save(str(EMBEDDINGS_NEW), embeddings)
    print(f"  ✓ Embeddings computed and saved: {embeddings.shape}")
else:
    print(f"  ✓ Using cached embeddings: {embeddings.shape}")


In [ ]:
# ── CELL 5 : GERMAN STOPWORDS ────────────────────────────────────────────────
# BERTopic needs German stopwords to avoid function words dominating topics.

import spacy

print("Loading German stopwords via spaCy...")
try:
    nlp = spacy.load('de_core_news_lg')
except OSError:
    print("  Downloading de_core_news_lg...")
    os.system('python -m spacy download de_core_news_lg')
    nlp = spacy.load('de_core_news_lg')

german_stopwords = list(nlp.Defaults.stop_words)

# Add domain-specific stopwords that are too generic for topic keywords
domain_stopwords = [
    'dass', 'auch', 'wird', 'sind', 'wurde', 'wurden', 'haben',
    'mehr', 'beim', 'vom', 'zur', 'zum', 'ab', 'bis', 'noch',
    'neue', 'neuen', 'neues', 'neue', 'ersten', 'erste', 'ersten',
    'rund', 'etwa', 'bereits', 'seit', 'immer', 'weiter', 'weiterhin',
    'dabei', 'dazu', 'damit', 'zudem', 'jedoch', 'allerdings', 'zwar',
    'soll', 'sollen', 'solle', 'sollte', 'kann', 'können', 'könnte',
    'müssen', 'muss', 'müsse', 'werde', 'werden', 'wären',
]
all_stopwords = list(set(german_stopwords + domain_stopwords))
print(f"  ✓ {len(all_stopwords)} stopwords loaded")


In [ ]:
# ── CELL 6 : RUN BERTOPIC ────────────────────────────────────────────────────

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

print("Configuring BERTopic components...")

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors    = 15,
    n_components   = 5,
    min_dist       = 0.0,
    metric         = 'cosine',
    random_state   = RANDOM_STATE,
)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    min_cluster_size  = MIN_TOPIC_SIZE,
    min_samples       = 5,
    metric            = 'euclidean',
    cluster_selection_method = 'eom',
    prediction_data   = True,
)

# CountVectorizer with German stopwords
vectorizer = CountVectorizer(
    stop_words  = all_stopwords,
    min_df      = 3,
    max_df      = 0.90,
    ngram_range = (1, 2),
)

# BERTopic model
topic_model = BERTopic(
    umap_model       = umap_model,
    hdbscan_model    = hdbscan_model,
    vectorizer_model = vectorizer,
    nr_topics        = NR_TOPICS,
    language         = 'german',
    calculate_probabilities = True,
    verbose          = True,
)

print(f"\nFitting BERTopic on {len(texts)} articles...")
print(f"  Using pre-computed embeddings: {embeddings.shape}")

topics, probs = topic_model.fit_transform(texts, embeddings)

topic_info = topic_model.get_topic_info()
n_topics   = len(topic_info[topic_info['Topic'] != -1])
n_outliers = sum(1 for t in topics if t == -1)

print(f"\n✓ BERTopic complete")
print(f"  Topics found    : {n_topics}")
print(f"  Outliers (-1)   : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
print(f"  Non-outliers    : {len(topics) - n_outliers}")


In [ ]:
# ── CELL 7 : DISPLAY ALL TOPICS AND KEYWORDS ─────────────────────────────────
# This is the primary output needed for taxonomy definition.
# Read these carefully to assign human-readable category labels.

print("=" * 70)
print("ALL TOPICS — TOP 15 KEYWORDS EACH")
print("(Used to define the classification taxonomy)")
print("=" * 70)

topic_keywords = {}
for _, row in topic_info.sort_values('Topic').iterrows():
    tid   = row['Topic']
    count = row['Count']
    if tid == -1:
        print(f"\n  Topic  -1  [{count:4d} articles]  OUTLIER — no coherent topic")
        topic_keywords[-1] = []
        continue
    words     = topic_model.get_topic(tid)
    kws_full  = [(w, round(s, 4)) for w, s in words[:15]]
    kws_short = [w for w, s in words[:10]]
    topic_keywords[tid] = kws_short
    print(f"\n  Topic {tid:2d}  [{count:4d} articles]")
    print(f"  Keywords: {', '.join(kws_short)}")

print("\n" + "=" * 70)
print("TOPIC SIZE SUMMARY")
print("=" * 70)
for _, row in topic_info.sort_values('Count', ascending=False).iterrows():
    tid   = int(row['Topic'])
    count = int(row['Count'])
    bar   = '█' * (count // 10)
    label = 'OUTLIER' if tid == -1 else f'Topic {tid:2d}'
    print(f"  {label:<10} {count:4d} articles  {bar}")


In [ ]:
# ── CELL 8 : TOPIC QUALITY METRICS ───────────────────────────────────────────

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

print("Computing topic coherence (C_v)...")

# Prepare data for gensim
tokenized = [text.lower().split() for text in texts]
dictionary = Dictionary(tokenized)

# Get topic keywords for coherence computation
topics_for_coherence = []
for tid in sorted(topic_keywords.keys()):
    if tid == -1:
        continue
    kws = [w for w in topic_keywords[tid] if w in dictionary.token2id]
    if len(kws) >= 3:
        topics_for_coherence.append(kws)

try:
    cm = CoherenceModel(
        topics     = topics_for_coherence,
        texts      = tokenized,
        dictionary = dictionary,
        coherence  = 'c_v',
    )
    coherence_cv = round(cm.get_coherence(), 4)
    print(f"  C_v coherence : {coherence_cv}")
except Exception as e:
    coherence_cv  = None
    print(f"  C_v coherence : could not compute ({e})")

# Topic diversity — proportion of unique words in top-10 keywords
all_kws   = [w for tid, kws in topic_keywords.items()
             if tid != -1 for w in kws[:10]]
diversity = round(len(set(all_kws)) / len(all_kws), 4) if all_kws else 0
print(f"  Diversity     : {diversity}")
print(f"  Topics        : {n_topics}")
print(f"  Outlier rate  : {n_outliers/len(topics)*100:.1f}%")


In [ ]:
# ── CELL 9 : SAVE ALL OUTPUTS ────────────────────────────────────────────────

# ── 1. Topic assignments CSV ──────────────────────────────────────────────────
# Clean column names — no 'bertopic_' prefix, consistent with downstream use

assignments = []
for i, (aid, tid, prob) in enumerate(zip(article_ids, topics, probs)):
    row_df = pipeline_df[pipeline_df['article_id'] == aid].iloc[0]

    # Get top-10 keywords as label string
    if tid == -1:
        topic_label = 'OTHER'
        topic_prob  = 0.0
    else:
        kws         = topic_keywords.get(tid, [])[:5]
        topic_label = '_'.join(kws) if kws else f'topic_{tid}'
        topic_prob  = round(float(max(prob)) if hasattr(prob, '__len__') else float(prob), 4)

    assignments.append({
        'article_id'  : aid,
        'source'      : row_df['source'],
        'year'        : int(row_df['year']),
        'year_bin'    : row_df['year_bin'],
        'topic'       : int(tid),
        'topic_label' : topic_label,
        'topic_prob'  : topic_prob,
        'is_outlier'  : tid == -1,
        'in_val_set'  : aid in VAL_IDS,
    })

assignments_df = pd.DataFrame(assignments)
assignments_df.to_csv(ASSIGNMENTS_OUT, index=False)
print(f"✓ Saved topic assignments: {ASSIGNMENTS_OUT.name}")
print(f"  Shape   : {assignments_df.shape}")
print(f"  Columns : {list(assignments_df.columns)}")

# ── 2. BERTopic model ────────────────────────────────────────────────────────
# Save as single file — use os.unlink to delete if exists (not shutil.rmtree)
if MODEL_OUT.exists():
    os.unlink(str(MODEL_OUT))
topic_model.save(str(MODEL_OUT), serialization='safetensors',
                 save_ctfidf=True, save_embedding_model=False)
print(f"✓ Saved BERTopic model: {MODEL_OUT.name}")

# ── 3. Summary JSON ───────────────────────────────────────────────────────────
# Validation set coverage
val_assignments = assignments_df[assignments_df['in_val_set']]
val_topic_dist  = val_assignments['topic'].value_counts().sort_index().to_dict()

summary = {
    'notebook'        : '06b_bertopic_fresh',
    'generated_at'    : datetime.now().isoformat(),
    'corpus_size'     : len(texts),
    'n_topics'        : n_topics,
    'n_outliers'      : n_outliers,
    'outlier_rate'    : round(n_outliers/len(topics), 4),
    'coherence_cv'    : coherence_cv,
    'diversity'       : diversity,
    'topic_keywords'  : {str(k): v for k, v in topic_keywords.items()
                         if k != -1},
    'topic_sizes'     : {str(int(row['Topic'])): int(row['Count'])
                         for _, row in topic_info.iterrows()
                         if int(row['Topic']) != -1},
    'validation_set'  : {
        'n_articles'  : len(val_assignments),
        'n_outliers'  : int((val_assignments['topic'] == -1).sum()),
        'n_with_topic': int((val_assignments['topic'] != -1).sum()),
        'topic_dist'  : {str(k): int(v) for k, v in val_topic_dist.items()},
    },
}

with open(SUMMARY_OUT, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved summary: {SUMMARY_OUT.name}")


In [ ]:
# ── CELL 10 : VALIDATION SET ANALYSIS ────────────────────────────────────────
# Shows exactly which topics appear in the 183-article validation set
# and how many articles per topic — important for downstream classification.

val_df = assignments_df[assignments_df['in_val_set']].copy()

print("=" * 70)
print(f"VALIDATION SET TOPIC DISTRIBUTION ({len(val_df)} articles)")
print("=" * 70)
print(f"\n  Outliers (OTHER)  : {(val_df['topic'] == -1).sum()}")
print(f"  With topic        : {(val_df['topic'] != -1).sum()}")

print(f"\n  Topic distribution:")
print(f"  {'Topic':>6}  {'Count':>6}  Top keywords")
print(f"  {'-'*60}")
for tid, count in val_df['topic'].value_counts().sort_index().items():
    if tid == -1:
        print(f"  {'OTHER':>6}  {count:>6}  (outlier — no coherent topic)")
        continue
    kws = ', '.join(topic_keywords.get(tid, [])[:6])
    print(f"  {tid:>6}  {count:>6}  {kws}")

# ── Topics with too few validation articles (problematic for classification) ──
print(f"\n  Topics with < 3 validation articles (may be too sparse):")
sparse = val_df[val_df['topic'] != -1]['topic'].value_counts()
sparse_topics = sparse[sparse < 3]
if len(sparse_topics):
    for tid, cnt in sparse_topics.items():
        print(f"    Topic {tid}: {cnt} articles")
else:
    print(f"    None — all topics have ≥ 3 validation articles")


In [ ]:
# ── CELL 11 : TOPIC OVERVIEW FIGURE ─────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A — topic sizes (full corpus)
topic_sizes = topic_info[topic_info['Topic'] != -1].sort_values(
    'Count', ascending=False)
axes[0].barh(
    [f"T{int(row['Topic'])}" for _, row in topic_sizes.iterrows()],
    topic_sizes['Count'],
    color='#2ca02c', edgecolor='black', linewidth=0.4, alpha=0.85
)
axes[0].axvline(x=MIN_TOPIC_SIZE, color='red', linestyle='--',
                linewidth=0.8, label=f'Min size ({MIN_TOPIC_SIZE})')
axes[0].set_xlabel('Article count', fontsize=10)
axes[0].set_title(f'Topic Sizes — Full Corpus ({n_topics} topics)',
                  fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(axis='x', alpha=0.3)

# Panel B — validation set topic distribution
val_counts = val_df['topic'].value_counts().sort_index()
colors_b   = ['#d62728' if t == -1 else '#2ca02c'
              for t in val_counts.index]
labels_b   = ['OTHER' if t == -1 else f'T{t}'
              for t in val_counts.index]
axes[1].bar(labels_b, val_counts.values,
            color=colors_b, edgecolor='black', linewidth=0.4, alpha=0.85)
axes[1].set_xlabel('Topic', fontsize=10)
axes[1].set_ylabel('Article count', fontsize=10)
axes[1].set_title(f'Validation Set Topic Distribution (n={len(val_df)})',
                  fontsize=11)
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(FIGURES_DIR / f'nb06b_topic_overview.{ext}',
                dpi=300 if ext == 'pdf' else 150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: nb06b_topic_overview.pdf/.png")


In [ ]:
# ── CELL 12 : FINAL SUMMARY ──────────────────────────────────────────────────

print("=" * 70)
print("NOTEBOOK 06b COMPLETE")
print("=" * 70)
print(f"  Topics found     : {n_topics}")
print(f"  Outlier rate     : {n_outliers/len(topics)*100:.1f}%")
print(f"  C_v coherence    : {coherence_cv}")
print(f"  Diversity        : {diversity}")
print()
print(f"  Validation set coverage:")
print(f"    Total articles : {len(val_df)}")
print(f"    With topic     : {(val_df['topic'] != -1).sum()}")
print(f"    Outliers       : {(val_df['topic'] == -1).sum()}")
print()
print(f"  Outputs saved:")
print(f"    {ASSIGNMENTS_OUT.name}")
print(f"    {MODEL_OUT.name}")
print(f"    {EMBEDDINGS_NEW.name}")
print(f"    {SUMMARY_OUT.name}")
print()
print("=" * 70)
print("NEXT STEP — TAXONOMY DEFINITION")
print("=" * 70)
print("Read the topic keywords in Cell 7 above.")
print("Share the full Cell 7 output so the taxonomy can be defined")
print("for Notebook 15 (topic classification comparison).")
print()
print("ALL TOPICS AND KEYWORDS FOR TAXONOMY:")
print("-" * 70)
for tid in sorted(topic_keywords.keys()):
    if tid == -1: continue
    kws   = ', '.join(topic_keywords[tid][:10])
    count = int(topic_info[topic_info['Topic'] == tid]['Count'].values[0])
    print(f"  Topic {tid:2d} [{count:4d}]: {kws}")
